# Tablr — Paid Acquisition & Creative Performance Analysis

**[SYNTHETIC DATA — Tablr portfolio simulation]**  
**Period:** 2026-01-05 to 2026-03-29 (12 weeks)  
**Analyst:** Phase 5, AI Growth Operations Agent  

This notebook answers the business question: *What happened in paid acquisition, which creative attributes explain it, and where should the team investigate further?*

All data is synthetic. Patterns were designed into the simulation to surface real analytical skills — not to represent actual market outcomes.

## 1. Setup & Data Loading

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

# Resolve data directory relative to this notebook
DATA_DIR = Path("__file__").parent.parent / "data" / "synthetic"
# Use absolute path for robustness
DATA_DIR = Path.cwd().parent / "data" / "synthetic"
if not DATA_DIR.exists():
    DATA_DIR = Path("../data/synthetic")

PERF = str(DATA_DIR / "fact_daily_performance.parquet")
DIMS = str(DATA_DIR / "dim_creatives.parquet")
TRIALS = str(DATA_DIR / "dim_trial_accounts.parquet")
EVENTS = str(DATA_DIR / "fact_product_events.parquet")
SUBS = str(DATA_DIR / "fact_subscriptions.parquet")

con = duckdb.connect()

# Quick sanity check
row_counts = con.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM read_parquet('{PERF}'))    AS perf_rows,
        (SELECT COUNT(*) FROM read_parquet('{DIMS}'))    AS creative_rows,
        (SELECT COUNT(*) FROM read_parquet('{TRIALS}'))  AS trial_rows,
        (SELECT COUNT(*) FROM read_parquet('{EVENTS}'))  AS event_rows
""").df()

print("[SYNTHETIC DATA] Row counts:")
row_counts

## 2. Channel Scorecard

**Key methodological note:** All rate metrics (CTR, CPM, CPC) use *weighted aggregation* — `SUM(clicks) / SUM(impressions)` — not the arithmetic mean of per-row rates. Averaging per-row rates gives incorrect results when rows have unequal impression counts (a form of Simpson's paradox risk).

In [ ]:
scorecard = con.execute(f"""
    SELECT
        channel,
        SUM(impressions)                                               AS impressions,
        SUM(clicks)                                                    AS clicks,
        ROUND(SUM(spend_usd), 2)                                      AS spend_usd,
        SUM(trial_signups)                                             AS trial_signups,
        ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3)  AS ctr_pct,
        ROUND(SUM(spend_usd) * 1000.0 / NULLIF(SUM(impressions), 0), 2) AS cpm_usd,
        ROUND(SUM(spend_usd) / NULLIF(SUM(clicks), 0), 2)            AS cpc_usd,
        ROUND(SUM(spend_usd) / NULLIF(SUM(trial_signups), 0), 2)     AS trial_cac_usd
    FROM read_parquet('{PERF}')
    GROUP BY channel
    ORDER BY spend_usd DESC
""").df()

scorecard

### Observations

- **Google Search** has the highest CTR (4.01%) by a wide margin — consistent with intent-based search; users self-select to click.
- **LinkedIn** spent $80K and produced **zero** trial signups in the platform data. Its CPM ($122.54) is high and its CPC ($17.44) is the most expensive of any channel. LinkedIn may be reaching the right job titles but has no direct conversion path in the current funnel.
- **TikTok** has the lowest CPM ($53.52), lowest CPC ($3.33), and lowest Trial CAC ($175.23) — making it the most cost-efficient channel for generating trial signups.
- **META** and **Google Search** spent nearly identical amounts (~$621K each) but Google's CTR is 2× META's. This reflects audience intent differences, not necessarily creative quality differences.

## 3. CPAO by Channel (Cost Per Activated Owner)

Trial signup is a weak conversion signal. The more meaningful metric is CPAO — what it cost to produce an owner who actually launched a Tablr marketing campaign within 14 days of signing up.

In [ ]:
cpao = con.execute(f"""
    WITH signups AS (
        SELECT account_id, CAST(event_timestamp AS TIMESTAMP) AS signup_ts
        FROM read_parquet('{EVENTS}')
        WHERE event_type = 'TRIAL_SIGNUP'
    ),
    activations AS (
        SELECT account_id, CAST(event_timestamp AS TIMESTAMP) AS activated_ts
        FROM read_parquet('{EVENTS}')
        WHERE event_type = 'CAMPAIGN_LAUNCHED'
    ),
    activated_14d AS (
        SELECT DISTINCT s.account_id
        FROM signups s
        JOIN activations a ON s.account_id = a.account_id
        WHERE DATEDIFF('day', CAST(s.signup_ts AS DATE), CAST(a.activated_ts AS DATE)) BETWEEN 0 AND 14
    ),
    trial_accts AS (
        SELECT account_id, channel
        FROM read_parquet('{TRIALS}')
        WHERE NOT is_missing_attribution
    ),
    activ_by_ch AS (
        SELECT t.channel,
               COUNT(DISTINCT t.account_id)   AS attributed_trials,
               COUNT(DISTINCT a14.account_id) AS activated_owners,
               ROUND(
                   COUNT(DISTINCT a14.account_id) * 100.0
                   / NULLIF(COUNT(DISTINCT t.account_id), 0), 2
               )                              AS activation_rate_pct
        FROM trial_accts t
        LEFT JOIN activated_14d a14 ON t.account_id = a14.account_id
        GROUP BY t.channel
    ),
    spend AS (
        SELECT channel, ROUND(SUM(spend_usd), 2) AS total_spend,
               SUM(trial_signups) AS platform_signups
        FROM read_parquet('{PERF}')
        GROUP BY channel
    )
    SELECT
        s.channel,
        s.total_spend,
        s.platform_signups,
        ROUND(s.total_spend / NULLIF(s.platform_signups, 0), 2)      AS trial_cac_usd,
        a.attributed_trials,
        a.activated_owners,
        a.activation_rate_pct,
        ROUND(s.total_spend / NULLIF(a.activated_owners, 0), 2)      AS cpao_usd
    FROM spend s
    LEFT JOIN activ_by_ch a ON s.channel = a.channel
    ORDER BY s.total_spend DESC
""").df()

cpao

### Observations

- The gap between **Trial CAC** and **CPAO** is 5–8x for every channel. This means most trial signups do not convert into activated owners within 14 days — the funnel leaks heavily between signup and first campaign launch.
- **TikTok** wins on both metrics: lowest Trial CAC ($175.23) and lowest CPAO ($908.48). Its activation rate (27.7%) is also the highest of the three measured channels.
- **META** has the highest activation-rate-adjusted cost ($1,396 CPAO), slightly above Google Search ($1,369), despite spending nearly the same amount.
- **LinkedIn** cannot be measured at CPAO level — it has no platform-reported signups and its attributed first-party trial count is too small. This is a data gap, not a finding.
- **Caution:** The activation window is 14 days. Owners who launched a campaign on day 15+ are classified as not activated. A longer window would increase activation counts and reduce CPAO.

## 4. Creative Performance by Hook Type

Hook type is the opening pattern of an ad (how it captures attention in the first 2–3 seconds). Understanding which hook patterns drive both clicks and conversions helps the creative team prioritize future production.

In [ ]:
hook_perf = con.execute(f"""
    SELECT
        dc.hook_type,
        SUM(f.impressions)                                                    AS impressions,
        SUM(f.clicks)                                                         AS clicks,
        ROUND(SUM(f.clicks) * 100.0 / NULLIF(SUM(f.impressions), 0), 3)     AS ctr_pct,
        ROUND(SUM(f.spend_usd), 2)                                           AS spend_usd,
        SUM(f.trial_signups)                                                  AS trial_signups,
        ROUND(SUM(f.spend_usd) / NULLIF(SUM(f.trial_signups), 0), 2)        AS trial_cac_usd
    FROM read_parquet('{PERF}') f
    JOIN read_parquet('{DIMS}') dc ON f.creative_id = dc.creative_id
    GROUP BY dc.hook_type
    ORDER BY ctr_pct DESC
""").df()

hook_perf

### Observations

- **FEAR_OF_MISSING_OUT** and **OUTCOME** hooks achieve the highest CTRs (2.40% and 2.31%) but do not produce the lowest trial CAC. High CTR does not automatically imply high conversion intent.
- **CONTRAST** hooks have the 3rd-highest CTR (1.98%) but the **lowest trial CAC ($216.23)** — the best CTR-to-conversion efficiency in the set. This suggests CONTRAST messaging attracts a higher-intent audience.
- **DEMONSTRATION** hooks have the lowest CTR (1.67%) and the highest trial CAC ($295.50). Functional demos may require more context than a short ad can provide — or they may appeal to a more cautious audience.
- **Caution:** Hook type and channel are confounded. Google Search uses different creative formats than TikTok, and both the channel and the hook affect CTR independently. Isolating hook effect requires within-channel experiments.

## 5. High CTR vs Low Activation Finding

Some creatives attract clicks (top CTR quartile) but fail to produce activated restaurant owners (bottom activation quartile). These are the most misleading creatives to optimize toward if you only look at CTR.

In [ ]:
high_ctr_low_act = con.execute(f"""
    WITH creative_perf AS (
        SELECT
            f.creative_id, dc.channel, dc.hook_type, dc.creative_format,
            SUM(f.clicks) * 1.0 / NULLIF(SUM(f.impressions), 0) AS ctr,
            SUM(f.trial_signups) AS trial_signups
        FROM read_parquet('{PERF}') f
        JOIN read_parquet('{DIMS}') dc ON f.creative_id = dc.creative_id
        GROUP BY f.creative_id, dc.channel, dc.hook_type, dc.creative_format
    ),
    signups AS (
        SELECT account_id, CAST(event_timestamp AS TIMESTAMP) AS signup_ts
        FROM read_parquet('{EVENTS}') WHERE event_type = 'TRIAL_SIGNUP'
    ),
    launched AS (
        SELECT account_id, CAST(event_timestamp AS TIMESTAMP) AS activated_ts
        FROM read_parquet('{EVENTS}') WHERE event_type = 'CAMPAIGN_LAUNCHED'
    ),
    activated_14d AS (
        SELECT DISTINCT s.account_id
        FROM signups s
        JOIN launched l ON s.account_id = l.account_id
        WHERE DATEDIFF('day', CAST(s.signup_ts AS DATE), CAST(l.activated_ts AS DATE)) BETWEEN 0 AND 14
    ),
    creative_activ AS (
        SELECT
            t.creative_id,
            COUNT(DISTINCT t.account_id) AS total_trials,
            COUNT(DISTINCT a.account_id) AS activated_owners
        FROM read_parquet('{TRIALS}') t
        LEFT JOIN activated_14d a ON t.account_id = a.account_id
        GROUP BY t.creative_id
    ),
    combined AS (
        SELECT
            cp.creative_id, cp.channel, cp.hook_type, cp.creative_format,
            ROUND(cp.ctr * 100, 3) AS ctr_pct,
            cp.trial_signups,
            COALESCE(ca.activated_owners, 0)                             AS activated_owners,
            ROUND(
                COALESCE(ca.activated_owners, 0) * 100.0
                / NULLIF(ca.total_trials, 0), 2
            )                                                             AS activation_rate_pct,
            NTILE(4) OVER (ORDER BY cp.ctr DESC)                         AS ctr_quartile,
            NTILE(4) OVER (
                ORDER BY COALESCE(ca.activated_owners, 0) * 1.0
                    / NULLIF(ca.total_trials, 0) ASC NULLS FIRST
            )                                                             AS activation_quartile
        FROM creative_perf cp
        LEFT JOIN creative_activ ca ON cp.creative_id = ca.creative_id
    )
    SELECT creative_id, channel, hook_type, creative_format,
           ctr_pct, trial_signups, activated_owners, activation_rate_pct
    FROM combined
    WHERE ctr_quartile = 1 AND activation_quartile = 1
    ORDER BY ctr_pct DESC
""").df()

print(f"[SYNTHETIC DATA] Creatives: top CTR quartile AND bottom activation quartile: {len(high_ctr_low_act)}")
high_ctr_low_act

### Observations

- 2 creatives fall into the high-CTR / low-activation quadrant. These are the most dangerous creatives to optimize toward if CTR is used as the primary success metric.
- A creative that drives clicks but not activations may be:
  - Appealing to the wrong audience segment (curiosity, not intent to buy)
  - Creating expectation mismatch with the landing page or onboarding flow
  - Attracting non-owner personas (e.g., suppliers, job seekers) who click but do not sign up
- **Next step:** Review the creative messaging for these IDs and compare the onboarding dropout rate for accounts they sourced vs. the rest of the portfolio.

## 6. Creative Fatigue Detection

Creative fatigue occurs when a creative's CTR peaks early and then declines significantly as the audience has already been exposed to the ad. We define fatigue as a ≥25% relative CTR drop from the peak week to the latest observed week, for creatives with at least 4 weeks of run time.

In [ ]:
fatigue = con.execute(f"""
    WITH min_date AS (
        SELECT MIN(CAST(date AS DATE)) AS start_date
        FROM read_parquet('{PERF}')
    ),
    daily_with_week AS (
        SELECT
            f.creative_id,
            dc.channel,
            dc.hook_type,
            CAST(DATEDIFF('day', m.start_date, CAST(f.date AS DATE)) / 7 AS INTEGER) AS week_num,
            f.clicks,
            f.impressions
        FROM read_parquet('{PERF}') f
        JOIN read_parquet('{DIMS}') dc ON f.creative_id = dc.creative_id
        CROSS JOIN min_date m
    ),
    weekly_ctr AS (
        SELECT creative_id, channel, hook_type, week_num,
               SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS ctr
        FROM daily_with_week
        GROUP BY creative_id, channel, hook_type, week_num
    ),
    peak AS (
        SELECT creative_id, channel, hook_type,
               MAX(ctr)              AS peak_ctr,
               MAX_BY(week_num, ctr) AS peak_week
        FROM weekly_ctr
        GROUP BY creative_id, channel, hook_type
    ),
    latest AS (
        SELECT creative_id,
               MAX_BY(ctr, week_num) AS latest_ctr,
               MAX(week_num)         AS latest_week
        FROM weekly_ctr
        GROUP BY creative_id
    )
    SELECT
        p.creative_id,
        p.channel,
        p.hook_type,
        p.peak_week,
        ROUND(p.peak_ctr * 100, 3)                                          AS peak_ctr_pct,
        l.latest_week,
        ROUND(l.latest_ctr * 100, 3)                                        AS latest_ctr_pct,
        ROUND((p.peak_ctr - l.latest_ctr) / NULLIF(p.peak_ctr, 0) * 100, 1) AS ctr_drop_pct,
        l.latest_week >= 3
            AND (p.peak_ctr - l.latest_ctr) / NULLIF(p.peak_ctr, 0) > 0.25  AS is_fatigued
    FROM peak p
    JOIN latest l ON p.creative_id = l.creative_id
    WHERE l.latest_week >= 3
    ORDER BY ctr_drop_pct DESC
""").df()

n_fatigued = fatigue[fatigue["is_fatigued"] == True].shape[0]
n_total = fatigue.shape[0]
print(f"[SYNTHETIC DATA] Fatigued creatives: {n_fatigued} of {n_total} ({n_fatigued/n_total*100:.1f}%)")

# Show top 10 by CTR drop
fatigue.head(10)

In [ ]:
# Summary: fatigued vs healthy by channel
fatigue_summary = fatigue.groupby(["channel", "is_fatigued"]).size().reset_index(name="count")
fatigue_summary

### Observations

- **16 of 90 creatives (17.8%)** show fatigue — defined as ≥25% relative CTR drop from peak to latest week.
- The three most severely fatigued creatives are all in **Google Search** (cr_google_026_001/002/003), each showing >91% CTR drop from a week-6 peak to week 12. This is an extreme signal — these creatives were likely over-served to a limited search audience.
- **META** and **TikTok** also have fatigued creatives, with drops in the 25–52% range.
- **Important caveat:** CTR decline alone does not prove fatigue. Budget reallocation, audience expansion, or external seasonality can also depress CTR. Frequency data and reach data should be checked before pausing a creative.
- **Practical implication:** Creatives that peaked in weeks 1–2 and ran for 12 weeks without refresh are the highest-priority refresh candidates.

## 7. Data Quality Summary

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))

from growth_agent.analytics.data_quality import run_all_checks, DATA_DIR as DQ_DATA_DIR

findings = run_all_checks(DATA_DIR)

dq_df = pd.DataFrame([
    {
        "check": f.check_name,
        "table": f.table,
        "severity": f.severity,
        "count": f.count,
        "description": f.description,
    }
    for f in findings
])

print("[SYNTHETIC DATA] Data quality findings:")
dq_df

### Observations

- **Missing attribution (8.2%, 409 accounts):** The most impactful quality issue. These accounts cannot be attributed to a channel, so they are excluded from CPAO calculations. If their activation rate differs from attributed accounts, all channel CPAOs are biased.
- **Clicks > impressions (9 rows — ERROR):** Nine rows in fact_daily_performance have more clicks than impressions, which is physically impossible. These rows should be investigated at the platform API level before being included in any spend calculation.
- **LinkedIn signups = 0 (INFO):** Confirmed — LinkedIn shows zero trial signups in fact_daily_performance. This is an intentional synthetic pattern reflecting a channel without a direct conversion path.
- All other checks (negative spend, duplicates, orphans) are clean.

## 8. Key Findings and Next Questions

### Summary of findings

| # | Finding | Confidence | Type |
|---|---------|------------|------|
| 1 | TikTok has the lowest Trial CAC ($175) and lowest CPAO ($908) | High (descriptive) | Descriptive |
| 2 | LinkedIn spent $80K and produced 0 platform trial signups | High (data fact) | Descriptive |
| 3 | CONTRAST hook type produces lowest trial CAC ($216), DEMONSTRATION the highest ($296) | Medium (confounded by channel) | Descriptive |
| 4 | 16/90 creatives show >25% CTR decline from peak — possible fatigue | Medium (CTR-based only) | Descriptive |
| 5 | 2 creatives are high-CTR but low-activation — misleading optimization targets | Medium | Descriptive |
| 6 | 8.2% missing attribution biases all per-channel CPAO calculations | High | Data quality |

### Next questions to investigate

1. **LinkedIn:** Is LinkedIn producing pipeline through a different mechanism (e.g., brand search uplift on Google) that would show up as Google-attributed conversions? A media mix model or holdout test would be needed.
2. **TikTok scalability:** If TikTok CPAO is lowest, what is the audience saturation threshold? How does CPAO change at 2× the current budget?
3. **Creative fatigue recovery:** For the 3 Google Search creatives with >91% CTR drop — have they been refreshed? What was the new creative's peak CTR?
4. **High-CTR / low-activation creatives:** What is the onboarding dropout step for accounts sourced from these creatives? Is the drop at profile creation, or earlier?
5. **Attribution recovery:** Can UTM parameters be reconstructed from server-side logs for the 409 missing-attribution accounts? Their activation rate compared to attributed accounts would reveal whether missing attribution is random or correlated with channel.
6. **Subscription conversion:** This analysis stopped at activation (CAMPAIGN_LAUNCHED). What is the CPAS (Cost Per Acquired Subscriber) by channel? TikTok's advantage at trial CAC may or may not persist through to revenue.

In [ ]:
# Final summary numbers for reference
summary = con.execute(f"""
    SELECT
        MIN(CAST(date AS DATE))  AS period_start,
        MAX(CAST(date AS DATE))  AS period_end,
        ROUND(SUM(spend_usd), 2) AS total_spend_usd,
        SUM(impressions)         AS total_impressions,
        SUM(clicks)              AS total_clicks,
        SUM(trial_signups)       AS total_trial_signups
    FROM read_parquet('{PERF}')
""").df()

print("[SYNTHETIC DATA] 12-week totals:")
summary